In [2]:
import os

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from matplotlib.lines import Line2D
from typing import Literal
from sklearn.decomposition import PCA

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats 

from support import prepare_count_matrix

In [3]:
METADATA_PATH = f"{os.getcwd()}/../../hgsoc_data_detective/"
DATA_PATH = f"{os.getcwd()}/../../alignment_results/"

In [45]:
df = pd.read_excel(METADATA_PATH + "samples_overview.xlsx", header=1, index_col=0)

In [46]:
df = df[["0509", "0626", "Site of origin"]]

In [47]:
df.drop(index=["Total across columns", 2205], inplace=True)

In [48]:
df.value_counts("Site of origin")

Site of origin
Unknown                   17
Omentum                   12
Ovary                      5
Para-aortic lymph node     1
Para-colic gutter          1
Name: count, dtype: int64

In [55]:
df = df.sort_index().sort_values("0509")

In [56]:
len(df)

36

In [57]:
sub_df = df[df["0509"]=="X"]
index_1 = ["230509/" + str(x) for x in sub_df.index]

sub_df = df[df["0626"]=="X"]
index_2 = ["230626/" + str(x) for x in sub_df.index]

In [58]:
len(index_1) + len(index_2)

36

In [59]:
df["Long Sample ID"] = sorted(index_1) + sorted(index_2)

In [64]:
df = df.drop(columns=["0509", "0626"])

In [65]:
df.head()

,Site of origin,Long Sample ID
Sample #,,
2018,Omentum,230509/2018
2023,Unknown,230509/2023
2094,Unknown,230509/2094
2126,Unknown,230509/2126
2129,Ovary,230509/2129


In [42]:
omentum_samples = list(df[df["Site of origin"]=="Omentum"].index)
omentum_samples = [str(x) for x in omentum_samples]
ovary_samples = list(df[df["Site of origin"]=="Ovary"].index)
ovary_samples = [str(x) for x in ovary_samples]

In [77]:
count_matrix = prepare_count_matrix(DATA_PATH)
# Select only bulk samples
count_matrix = count_matrix[~count_matrix.index.str.contains("2304")]
# Only consider genes that have more than 10 read counts in total
count_matrix = count_matrix[count_matrix.columns[count_matrix.sum(axis=0) >= 10]]
# Add site of origin 
count_matrix.index.name = "Long Sample ID"
count_df = pd.merge(df, count_matrix, how="inner", left_on="Long Sample ID", right_on="Long Sample ID")

In [79]:
count_df.head()

,Site of origin,Long Sample ID,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,...,ENSG00000291299,ENSG00000291300,ENSG00000291309,ENSG00000291317,ENSG00000291326,ENSG00000291342,ENSG00000292223,ENSG00000292246,ENSG00000292271,ENSG00000292309
0,Omentum,230509/2018,205,2,1629,81,78,615,1159,457,...,362,21,6,50,7,0,0,1,1,2
1,Unknown,230509/2023,224,11,1375,75,54,480,1960,329,...,268,13,6,28,13,2,0,3,0,16
2,Unknown,230509/2094,321,13,2669,61,59,508,3878,494,...,199,8,3,30,3,0,4,2,0,2
3,Unknown,230509/2126,292,3,1092,75,120,647,1336,663,...,195,4,7,59,5,0,1,2,0,1
4,Ovary,230509/2129,221,9,600,84,89,148,1821,233,...,132,35,4,60,2,0,0,2,0,0
